In [0]:
# =========================================================
# PHASE 3: DATA QUALITY AUDIT FRAMEWORK
# =========================================================

from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

from src.quality.checks import (
    validate_record_count,
    count_duplicate_keys,
    validate_unique_key
)


# =========================================================
# 1. CONFIGURATION
# =========================================================

SILVER_TABLE = (
    "workspace.video_game_sales.silver_video_game_sales"
)

AUDIT_TABLE = (
    "workspace.video_game_sales.data_quality_audit"
)

silver_df = spark.table(SILVER_TABLE)

run_id = str(uuid.uuid4())


# =========================================================
# 2. COLLECT DATA QUALITY METRICS
# =========================================================

total_records = silver_df.count()

valid_years = silver_df.filter(
    F.col("year_status") == "VALID"
).count()

invalid_years = silver_df.filter(
    F.col("year_status") == "MISSING_OR_INVALID"
).count()

sales_matches = silver_df.filter(
    F.col("sales_reconciliation_status") == "MATCH"
).count()

sales_differences = silver_df.filter(
    F.col("sales_reconciliation_status") == "DIFFERENCE_DETECTED"
).count()

null_years = silver_df.filter(
    F.col("year").isNull()
).count()


# =========================================================
# 3. REUSABLE DATA QUALITY VALIDATIONS
# =========================================================

# Record count validation
record_count_valid = validate_record_count(
    total_records,
    16598
)

# Rank duplicate analysis
duplicate_rank_groups = count_duplicate_keys(
    silver_df,
    "rank"
)

# Rank uniqueness validation
rank_unique = validate_unique_key(
    silver_df,
    "rank"
)


# =========================================================
# 4. DEFINE AUDIT CHECKS
# =========================================================

checks = [

    (
        "SILVER_RECORD_COUNT",
        "16598",
        str(total_records),
        "PASS" if record_count_valid else "FAIL"
    ),

    (
        "VALID_YEAR_COUNT",
        "16327",
        str(valid_years),
        "PASS" if valid_years == 16327 else "FAIL"
    ),

    (
        "INVALID_YEAR_COUNT",
        "271",
        str(invalid_years),
        "PASS" if invalid_years == 271 else "FAIL"
    ),

    (
        "SALES_MATCH_COUNT",
        "12087",
        str(sales_matches),
        "PASS" if sales_matches == 12087 else "FAIL"
    ),

    (
        "SALES_DIFFERENCE_COUNT",
        "4511",
        str(sales_differences),
        "PASS" if sales_differences == 4511 else "FAIL"
    ),

    (
        "NULL_YEAR_COUNT",
        "271",
        str(null_years),
        "PASS" if null_years == 271 else "FAIL"
    ),

    (
        "DUPLICATE_RANK_GROUPS",
        "0",
        str(duplicate_rank_groups),
        "PASS" if duplicate_rank_groups == 0 else "FAIL"
    ),

    (
        "RANK_UNIQUENESS",
        "True",
        str(rank_unique),
        "PASS" if rank_unique else "FAIL"
    )
]


# =========================================================
# 5. CREATE AUDIT RECORDS
# =========================================================

audit_records = [
    (
        run_id,
        "SILVER",
        SILVER_TABLE,
        check_name,
        expected,
        actual,
        status
    )
    for check_name, expected, actual, status in checks
]

audit_df = spark.createDataFrame(
    audit_records,
    [
        "run_id",
        "pipeline_layer",
        "table_name",
        "check_name",
        "expected_value",
        "actual_value",
        "status"
    ]
).withColumn(
    "execution_timestamp",
    F.current_timestamp()
)


# =========================================================
# 6. PERSIST AUDIT HISTORY
# =========================================================

(
    audit_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(AUDIT_TABLE)
)


# =========================================================
# 7. DISPLAY CURRENT AUDIT RESULTS
# =========================================================

print("AUDIT RUN ID:", run_id)

current_audit = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("run_id") == run_id)
    .orderBy("check_name")
)

display(current_audit)


# =========================================================
# 8. FAIL PIPELINE IF ANY CHECK FAILS
# =========================================================

failed_checks = sum(
    1 for check in checks
    if check[3] == "FAIL"
)

print("Total checks:", len(checks))
print("Passed checks:", len(checks) - failed_checks)
print("Failed checks:", failed_checks)

assert failed_checks == 0, (
    "DATA QUALITY VALIDATION FAILED"
)

print("DATA QUALITY AUDIT PASSED")


# =========================================================
# 9. AUDIT TABLE SCHEMA
# =========================================================

spark.sql(
    f"DESCRIBE TABLE {AUDIT_TABLE}"
).show(truncate=False)


# =========================================================
# 10. HISTORICAL AUDIT RESULTS
# =========================================================

print("DATA QUALITY AUDIT HISTORY")

display(
    spark.table(AUDIT_TABLE)
    .orderBy(
        F.col("execution_timestamp").desc()
    )
)


# =========================================================
# 11. AUDIT STATUS SUMMARY
# =========================================================

print("AUDIT STATUS SUMMARY")

display(
    spark.table(AUDIT_TABLE)
    .groupBy("status")
    .count()
)


# =========================================================
# 12. CHECK-LEVEL HISTORICAL SUMMARY
# =========================================================

print("CHECK-LEVEL AUDIT SUMMARY")

display(
    spark.table(AUDIT_TABLE)
    .groupBy("status", "check_name")
    .count()
    .orderBy("check_name")
)


# =========================================================
# DATA QUALITY FRAMEWORK COMPLETED
# =========================================================

print("=================================================")
print("DATA QUALITY FRAMEWORK EXECUTION COMPLETED")
print("=================================================")

In [0]:
spark.sql("""
DESCRIBE TABLE workspace.video_game_sales.data_quality_audit
""").show(truncate=False)

display(
    spark.table(
        "workspace.video_game_sales.data_quality_audit"
    ).orderBy(
        F.desc("execution_timestamp")
    )
)

In [0]:
spark.sql("""
DESCRIBE TABLE workspace.video_game_sales.data_quality_audit
""").show(truncate=False)

display(
    spark.table(
        "workspace.video_game_sales.data_quality_audit"
    ).orderBy(
        F.desc("execution_timestamp")
    )
)

In [0]:
audit_table = "workspace.video_game_sales.data_quality_audit"

display(
    spark.table(audit_table)
    .orderBy("execution_timestamp", ascending=False)
)

In [0]:
display(
    spark.table(audit_table)
    .groupBy("status")
    .count()
)

In [0]:
audit_df = spark.table("workspace.video_game_sales.data_quality_audit")

display(
    audit_df.groupBy("status", "check_name")
    .count()
    .orderBy("check_name")
)